# Olist Orders - DataFrame Schema (pandera)

Define e valida o schema do dataset `olist_orders_dataset.csv` usando `pandera`.

In [ ]:
import pandas as pd
import pandera.pandas as pa
from pandera.typing import Series

from module_olist.config import RAW_DATA_DIR

input_file = RAW_DATA_DIR / "olist_orders_dataset.csv"
df = pd.read_csv(input_file)
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


: 

## Schema (estilo classe)

In [2]:
ORDER_STATUSES = [
    "delivered",
    "invoiced",
    "shipped",
    "processing",
    "unavailable",
    "canceled",
    "created",
    "approved",
]


class OrdersSchema(pa.DataFrameModel):
    order_id: Series[str] = pa.Field(unique=True, nullable=False)
    customer_id: Series[str] = pa.Field(nullable=False)
    order_status: Series[str] = pa.Field(isin=ORDER_STATUSES, nullable=False)

    order_purchase_timestamp: Series[str] = pa.Field(nullable=False)
    order_approved_at: Series[str] = pa.Field(nullable=True)
    order_delivered_carrier_date: Series[str] = pa.Field(nullable=True)
    order_delivered_customer_date: Series[str] = pa.Field(nullable=True)
    order_estimated_delivery_date: Series[str] = pa.Field(nullable=False)

    class Config:
        strict = True
        coerce = True

In [3]:
validated_df = OrdersSchema.validate(df, lazy=True)
validated_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## Testando com dados inválidos

`lazy=True` acumula todos os erros de validação em vez de parar no primeiro, e levanta `SchemaErrors` com o relatório completo.

In [4]:
bad_df = df.copy()
bad_df.loc[0, "order_status"] = "status_invalido"
bad_df.loc[1, "order_id"] = bad_df.loc[2, "order_id"]  # duplicado

try:
    OrdersSchema.validate(bad_df, lazy=True)
except pa.errors.SchemaErrors as err:
    print(err.failure_cases)

  schema_context        column  \
0         Column      order_id   
1         Column      order_id   
2         Column  order_status   

                                               check check_number  \
0                                   field_uniqueness         None   
1                                   field_uniqueness         None   
2  isin(['delivered', 'invoiced', 'shipped', 'pro...            0   

                       failure_case  index  
0  47770eb9100c2d0c44946d9cf07ec65d      1  
1  47770eb9100c2d0c44946d9cf07ec65d      2  
2                   status_invalido      0  
